# Retrieval Dataset Analysis

Notebook for dataset-level inspection using the refactored `src/` package.

In [ ]:
import sys
from pathlib import Path

def detect_runtime_environment() -> str:
    try:
        import google.colab  # type: ignore  # noqa: F401
        return 'colab'
    except Exception:
        if Path('/kaggle/input').exists():
            return 'kaggle'
        return 'local'

def add_project_root_to_syspath(project_name: str = 'retrieval_project') -> None:
    runtime_env = detect_runtime_environment()
    candidates = [Path.cwd(), *Path.cwd().parents]
    if runtime_env == 'colab':
        drive_root = Path('/content/drive/MyDrive')
        if not drive_root.exists():
            from google.colab import drive  # type: ignore
            drive.mount('/content/drive', force_remount=False)
        candidates = [Path('/content'), Path('/content/drive/MyDrive'), Path('/content/drive/Shareddrives'), *candidates]
    elif runtime_env == 'kaggle':
        candidates = [Path('/kaggle/working'), *candidates]

    seen = set()
    for base in candidates:
        key = str(base)
        if key in seen:
            continue
        seen.add(key)
        if (base / 'src' / 'infra' / 'notebook.py').exists():
            sys.path.insert(0, str(base))
            return
        if (base / project_name / 'src' / 'infra' / 'notebook.py').exists():
            sys.path.insert(0, str(base / project_name))
            return
        if runtime_env == 'colab' and base.exists():
            for match in base.rglob(project_name):
                if (match / 'src' / 'infra' / 'notebook.py').exists():
                    sys.path.insert(0, str(match))
                    return
    raise FileNotFoundError('Could not locate project root containing src/infra/notebook.py')

add_project_root_to_syspath()

from src.infra.notebook import setup_notebook

runtime_env, project_root = setup_notebook()
print(f'Detected runtime: {runtime_env}')
print(f'Project root    : {project_root}')


In [ ]:
from src.config import DEFAULT_CONFIG
from src.evaluation import load_ground_truth
from src.pipeline import bootstrap, load_project_frames

paths, config = bootstrap()
frames = load_project_frames(paths, DEFAULT_CONFIG)
ground_truth = load_ground_truth(paths.data_dir / "qgts_train.json")

print(f"Documents     : {len(frames.docs):,}")
print(f"Train queries : {len(frames.train_queries):,}")
print(f"Test queries  : {len(frames.test_queries):,}")
print(f"Ground truth  : {len(ground_truth):,}")

## Category Distribution

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")

doc_counts = frames.docs["category"].value_counts().sort_values(ascending=True)
train_counts = frames.train_queries["category"].value_counts().sort_values(ascending=True)
test_counts = frames.test_queries["category"].value_counts().sort_values(ascending=True) if "category" in frames.test_queries.columns else pd.Series(dtype=int)

gt_by_cat = {}
for entry in ground_truth.values():
    cat = entry["category"]
    gt_by_cat.setdefault(cat, []).append(len(entry["relevant_doc_ids"]))

all_categories = sorted(set(doc_counts.index) | set(train_counts.index) | set(test_counts.index))
ratio_series = pd.Series({cat: doc_counts.get(cat, 0) / max(train_counts.get(cat, 0), 1) for cat in all_categories}).sort_values()

fig, axes = plt.subplots(2, 2, figsize=(20, 12))
fig.suptitle("Dataset Category Distribution", fontsize=16, fontweight="bold")

doc_colors = plt.colormaps["viridis"](np.linspace(0.2, 0.9, len(doc_counts)))
train_colors = plt.colormaps["coolwarm"](np.linspace(0.1, 0.9, len(train_counts)))

doc_counts.plot(kind="barh", ax=axes[0, 0], color=doc_colors)
axes[0, 0].set_title("Documents by category", fontweight="bold")
axes[0, 0].set_xlabel("Count")

train_counts.plot(kind="barh", ax=axes[0, 1], color=train_colors)
axes[0, 1].set_title("Train queries by category", fontweight="bold")
axes[0, 1].set_xlabel("Count")

ratio_series.plot(kind="barh", ax=axes[1, 0], color="steelblue")
axes[1, 0].axvline(ratio_series.mean(), color="red", linestyle="--", linewidth=1.5, label=f"Mean = {ratio_series.mean():.1f}")
axes[1, 0].legend()
axes[1, 0].set_title("Docs per train-query ratio", fontweight="bold")
axes[1, 0].set_xlabel("Docs / train query")

coverage = pd.DataFrame(
    {
        "Total docs": [doc_counts.get(cat, 0) for cat in all_categories],
        "Train queries": [train_counts.get(cat, 0) for cat in all_categories],
        "Avg rel docs/query": [float(np.mean(gt_by_cat.get(cat, [0.0]))) for cat in all_categories],
    },
    index=all_categories,
)
norm = coverage.copy().astype(float)
for column in norm.columns:
    col_min = norm[column].min()
    col_max = norm[column].max()
    norm[column] = 0.5 if col_max == col_min else (norm[column] - col_min) / (col_max - col_min)

im = axes[1, 1].imshow(norm.values, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)
axes[1, 1].set_title("Category coverage overview", fontweight="bold")
axes[1, 1].set_xticks(range(len(coverage.columns)))
axes[1, 1].set_xticklabels(list(coverage.columns), rotation=0)
axes[1, 1].set_yticks(range(len(coverage.index)))
axes[1, 1].set_yticklabels(list(coverage.index))
for row_idx, category in enumerate(coverage.index):
    for col_idx, column in enumerate(coverage.columns):
        value = coverage.iloc[row_idx, col_idx]
        label = f"{value:.1f}" if isinstance(value, float) and not float(value).is_integer() else f"{int(value):,}"
        color = "white" if norm.iloc[row_idx, col_idx] > 0.6 else "black"
        axes[1, 1].text(col_idx, row_idx, label, ha="center", va="center", color=color, fontsize=8)
fig.colorbar(im, ax=axes[1, 1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## Basic Statistics

In [ ]:
summary_df = pd.DataFrame(
    {
        "documents": doc_counts,
        "train_queries": train_counts,
        "test_queries": pd.Series({cat: test_counts.get(cat, 0) for cat in all_categories}),
        "avg_rel_docs_per_train_query": pd.Series({cat: float(np.mean(gt_by_cat.get(cat, [0.0]))) for cat in all_categories}),
    }
).fillna(0)
summary_df["docs_per_train_query"] = summary_df["documents"] / summary_df["train_queries"].clip(lower=1)
summary_df = summary_df.sort_values("documents", ascending=False)
summary_df

In [ ]:
print(f"Average document length (chars): {frames.docs['content'].str.len().mean():.1f}")
print(f"Average train query length      : {frames.train_queries['content'].str.len().mean():.1f}")
print(f"Average test query length       : {frames.test_queries['content'].str.len().mean():.1f}")
print(f"Number of categories           : {len(all_categories)}")